# Tuning MLP 3-?-1 — Cross-Validation + GridSearchCV (Opsi 1)

Notebook ini **melengkapi** `training_mlp_3input_closedloop.ipynb`, TIDAK menggantikannya.
Tujuannya menjawab: *"Apakah ada setting MLP yang lebih baik dari 3-4-1 sekarang?"* secara
**jujur** dan **bisa dipertanggungjawabkan** di sidang.

**Dua perbaikan metodologi:**
1. **K-Fold Cross-Validation** — bukan satu split. Pada data kecil (453 baris), satu split
   bisa menipu. CV merata-ratakan banyak split sehingga angkanya stabil.
2. **GridSearchCV** — mencoba banyak kombinasi (jumlah neuron, fungsi aktivasi, regularisasi
   `alpha`) secara otomatis, lalu memilih yang terbaik berdasarkan skor CV.

Di akhir, model terbaik dibandingkan adil dengan model lama (3-4-1, tanh) di data uji yang sama.

## 1. Import library

In [1]:
import os, glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)

## 2. Muat & rekayasa fitur (SAMA PERSIS dengan notebook utama)
Agar perbandingan adil, pemuatan data, pembentukan fitur, dan target dibuat identik dengan
`training_mlp_3input_closedloop.ipynb`.

In [2]:
DATA_DIR = r'E:/SEMESTER 8/TA/BUKU TA_YOEL/DATA TRAINING 14 JUNI'
INCLUDE_F1   = True
RANDOM_STATE = 42
TEST_SIZE    = 0.20
FEATURES = ['error', 'd_error', 'duty_percent']
TARGET   = 'delta_duty'

KEY = ['pressure_bar', 'duty_percent', 'setpoint_bar', 'episode_id', 'is_decision']
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
parts, gid = [], 0
for f in files:
    d = pd.read_csv(f)
    for c in KEY:
        d[c] = pd.to_numeric(d[c], errors='coerce')
    d = d.dropna(subset=KEY)
    for ep in sorted(d['episode_id'].unique()):
        gid += 1
        sub = d[d['episode_id'] == ep].copy()
        sub['global_episode'] = gid
        parts.append(sub)
df = pd.concat(parts, ignore_index=True)

dec = df[df['is_decision'] == 1].copy().reset_index(drop=True)
dec['error']      = dec['setpoint_bar'] - dec['pressure_bar']
dec['d_error']    = dec.groupby('global_episode')['error'].diff().fillna(0.0)
dec['delta_duty'] = dec.groupby('global_episode')['duty_percent'].diff().shift(-1)
data = dec.dropna(subset=['delta_duty']).reset_index(drop=True)
if not INCLUDE_F1:
    data = data[np.isclose(data['setpoint_bar'], 0.30)].reset_index(drop=True)

X = data[FEATURES].to_numpy(np.float32)
y = data[TARGET].to_numpy(np.float32)
print('Total baris latih:', len(data))

Total baris latih: 453


## 3. Referensi: model SAAT INI (3-4-1, tanh) dengan 5-fold CV
Sebelum mencari yang lebih baik, kita ukur dulu model sekarang secara jujur memakai CV.
Inilah angka pembanding yang sebenarnya (biasanya sedikit di bawah angka satu-split di buku).

In [3]:
def make_pipe(hidden=(4,), activation='tanh', alpha=1e-3, solver='lbfgs'):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(hidden_layer_sizes=hidden, activation=activation,
                             solver=solver, alpha=alpha, max_iter=5000,
                             random_state=RANDOM_STATE, tol=1e-7)),
    ])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cur = make_pipe()
r2  = cross_val_score(cur, X, y, cv=cv, scoring='r2')
mae = -cross_val_score(cur, X, y, cv=cv, scoring='neg_mean_absolute_error')
print('Model SAAT INI (3-4-1, tanh, alpha=1e-3):')
print('  R2  per fold:', np.round(r2, 3), '| rata2 %.3f +/- %.3f' % (r2.mean(), r2.std()))
print('  MAE per fold:', np.round(mae, 3), '| rata2 %.3f +/- %.3f' % (mae.mean(), mae.std()))

Model SAAT INI (3-4-1, tanh, alpha=1e-3):
  R2  per fold: [0.925 0.864 0.835 0.919 0.928] | rata2 0.894 +/- 0.038
  MAE per fold: [0.351 0.404 0.444 0.357 0.34 ] | rata2 0.379 +/- 0.039


## 4. GridSearchCV — cari kombinasi terbaik
Kombinasi yang diuji:
- **jumlah neuron hidden**: 3, 4, 6, 8 (tetap 1 hidden layer)
- **fungsi aktivasi**: tanh, relu
- **alpha (regularisasi)**: 1e-4, 1e-3, 1e-2

Total 4 x 2 x 3 = 24 kombinasi, masing-masing dievaluasi 5-fold (120 pelatihan). `solver='lbfgs'`
dipertahankan karena paling stabil untuk data kecil. Skor pemilihan = R² rata-rata CV.

In [4]:
param_grid = {
    'mlp__hidden_layer_sizes': [(3,), (4,), (6,), (8,)],
    'mlp__activation': ['tanh', 'relu'],
    'mlp__alpha': [1e-4, 1e-3, 1e-2],
}
base = make_pipe()
grid = GridSearchCV(base, param_grid, cv=cv, scoring='r2', n_jobs=-1, refit=True)
grid.fit(X, y)
print('Kombinasi terbaik :', grid.best_params_)
print('R2 CV terbaik     : %.4f' % grid.best_score_)

Kombinasi terbaik : {'mlp__activation': 'tanh', 'mlp__alpha': 0.001, 'mlp__hidden_layer_sizes': (8,)}
R2 CV terbaik     : 0.9083


### 4b. 8 kombinasi teratas (untuk tabel laporan)

In [5]:
res = pd.DataFrame(grid.cv_results_)
cols = ['param_mlp__hidden_layer_sizes', 'param_mlp__activation', 'param_mlp__alpha',
        'mean_test_score', 'std_test_score', 'rank_test_score']
print(res[cols].sort_values('rank_test_score').head(8).to_string(index=False))

param_mlp__hidden_layer_sizes param_mlp__activation  param_mlp__alpha  mean_test_score  std_test_score  rank_test_score
                         (8,)                  tanh            0.0010         0.908271        0.039107                1
                         (8,)                  tanh            0.0100         0.906722        0.053352                2
                         (8,)                  relu            0.0010         0.906536        0.049511                3
                         (6,)                  tanh            0.0100         0.904703        0.046070                4
                         (6,)                  tanh            0.0010         0.901399        0.054069                5
                         (8,)                  tanh            0.0001         0.900684        0.052206                6
                         (8,)                  relu            0.0001         0.899970        0.063301                7
                         (3,)           

## 5. Uji adil di data test yang sama
Pakai pembagian train/test yang sama (random_state=42), latih ulang **model terbaik** vs
**model lama (3-4-1)** pada train yang sama, lalu bandingkan di test yang sama.

In [6]:
def metrics(yt, yp):
    return {'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAE':  mean_absolute_error(yt, yp),
            'R2':   r2_score(yt, yp)}

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True)

old = make_pipe().fit(Xtr, ytr)
best = grid.best_estimator_.__class__(grid.best_estimator_.steps)  # pipeline baru param terbaik
best = make_pipe(hidden=grid.best_params_['mlp__hidden_layer_sizes'],
                 activation=grid.best_params_['mlp__activation'],
                 alpha=grid.best_params_['mlp__alpha']).fit(Xtr, ytr)

rows = [{'Model': 'Lama (3-4-1, tanh)', **metrics(yte, old.predict(Xte))},
        {'Model': 'Terbaik (grid)',    **metrics(yte, best.predict(Xte))}]
print(pd.DataFrame(rows).round(4).to_string(index=False))

             Model   RMSE    MAE     R2
Lama (3-4-1, tanh) 0.4343 0.3509 0.9253
    Terbaik (grid) 0.5661 0.3251 0.8731


## 6. Kesimpulan
- Jika **"Terbaik (grid)" hampir sama** dengan model lama -> ini bukti ilmiah bahwa **3-4-1
  sudah cukup baik** (justifikasi kuat untuk sidang: pilihan sederhana bukan kebetulan).
- Jika **lebih baik secara konsisten** -> pertimbangkan ganti, lalu ekspor ulang bobot ke STM32.

Yang penting: angka CV (Bagian 3-4) lebih jujur daripada satu-split, sehingga klaim performa
lebih kuat dan tahan pertanyaan penguji.

## 7. (Opsional) Ekspor bobot model terbaik untuk STM32
Hanya berjalan jika model terbaik tetap 1 hidden layer (format firmware Anda).

In [7]:
sc = best.named_steps['scaler']; ml = best.named_steps['mlp']
hs = grid.best_params_['mlp__hidden_layer_sizes']
print('Arsitektur terbaik: 3-%d-1 | aktivasi=%s | alpha=%g'
      % (hs[0], grid.best_params_['mlp__activation'], grid.best_params_['mlp__alpha']))
print('scaler_mean :', np.round(sc.mean_, 6))
print('scaler_scale:', np.round(sc.scale_, 6))
print('W1 shape', ml.coefs_[0].shape, '| W2 shape', ml.coefs_[1].shape)
print('CATATAN: jika aktivasi terbaik = relu, ganti tanhf() -> fmaxf(0,x) di firmware.')

Arsitektur terbaik: 3-8-1 | aktivasi=tanh | alpha=0.001
scaler_mean : [-0.013598 -0.000125 82.190608]
scaler_scale: [0.114402 0.059285 7.22982 ]
W1 shape (3, 8) | W2 shape (8, 1)
CATATAN: jika aktivasi terbaik = relu, ganti tanhf() -> fmaxf(0,x) di firmware.
